In [1]:
import os
import requests
from typing import Tuple

def download_copernicus_dsm_30m(
    bbox: Tuple[float, float, float, float],
    output_path: str,
    api_key: str = None
) -> str:
    """
    Download Copernicus DEM (DSM) 30m for the given bbox.
    bbox: (minx, miny, maxx, maxy) in EPSG:4326
    output_path: local filename (GeoTIFF)
    api_token: optional token if required
    """
    minx, miny, maxx, maxy = bbox
    # Example endpoint – you’ll need to confirm exact param names from docs:
    url = "https://services.dataspace.copernicus.eu/process"
    payload = {
        "input": {
            "data": [
                {
                    "type": "DEM",
                    "dataFilter": {
                        "demInstance": "COPERNICUS_30",
                        "bbox": {
                            "west": minx,
                            "south": miny,
                            "east": maxx,
                            "north": maxy
                        }
                    }
                }
            ]
        },
        "output": {
            "format": {
                "type": "image/tiff"
            }
        },
        "processing": {
            "upsampling": "NEAREST",
            "downsampling": "NEAREST"
        }
    }
    headers = {}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    resp = requests.post(url, json=payload, headers=headers, stream=True)
    resp.raise_for_status()
    with open(output_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    return output_path

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.client import Config
import os

def download_copernicus_tiles_for_bbox(bbox, local_folder):
    """
    Downloads Copernicus DEM 30m tiles from S3 that overlap with bbox.
    bbox = (minx, miny, maxx, maxy) in EPSG:4326.
    """
    # example for 30m bucket
    bucket = "copernicus-dem-30m"
    s3 = boto3.client('s3', region_name='eu-central-1',
                      config=Config(signature_version=UNSIGNED))
    # list objects, pick keys overlapping your bbox (you need mapping tile key → bbox)
    response = s3.list_objects_v2(Bucket=bucket, Prefix="Copernicus_DSM_COG_10_")
    for obj in response.get('Contents', []):
        key = obj['Key']
        # You need logic to decide if this key’s tile overlaps your bbox
        # For demo: just download
        local_path = os.path.join(local_folder, os.path.basename(key))
        s3.download_file(bucket, key, local_path)
        print("Downloaded", key)


In [2]:
bbox = (6.85, 45.25, 6.90, 45.30)  # example around Rennes region (adapt as needed)
api_key = "38b3211483d503ddd81882f2259b6201"

cop_out = download_copernicus_dsm_30m(bbox, "copernicus_30m.tif", api_key="api_key")
print("Copernicus DSM downloaded:", cop_out)

ConnectionError: HTTPSConnectionPool(host='services.dataspace.copernicus.eu', port=443): Max retries exceeded with url: /process (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f062b673350>: Failed to resolve 'services.dataspace.copernicus.eu' ([Errno -2] Name or service not known)"))